# EDA — Lead Prioritization Dataset

**Goal:** understand `leads.csv`, surface the real data-quality issues, and extract the patterns that should drive feature-engineering, validation, and modeling decisions.

> **Note:** the task brief explicitly states that a Jupyter Notebook is *not* accepted as the final deliverable. This notebook is the *exploration* layer only; every conclusion here is mapped to a concrete pipeline decision in the final section.

## The problem in one view
A large number of leads enter the insurance purchase funnel every day, but most never complete the purchase. Telesales capacity is limited, so we cannot call everyone. The desired output is a **ranking** of leads by probability of completing the purchase (`Completed Purchase`) and by business value (`Expected Margin`).

The important framing: this is a **ranking problem under a capacity constraint**, not plain classification. The metrics have to measure that.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.dpi": 110, "figure.autolayout": True, "axes.grid": True,
                     "grid.alpha": .25, "axes.spines.top": False, "axes.spines.right": False})

DATA = Path("../data")
CHARTS = Path("../charts"); CHARTS.mkdir(exist_ok=True)
TARGET = "Completed Purchase"

def save(fig, name):
    fig.savefig(CHARTS / f"{name}.png", bbox_inches="tight")
    return fig

raw = pd.read_csv(DATA / "leads.csv")
raw["Created At"] = pd.to_datetime(raw["Created At"])
print(raw.shape)
raw.head(3)

(50180, 22)


,Lead ID,Created At,Product Type,Channel,Device,Partner,City,Insurance Company,Payment Type,Minutes Since Abandonment,Days To Policy Expiry,Price,Discount Percent,Has Previous Purchase,Visited Offer Page,Incoming Call Last 24h,Sessions Last 7d,Offer Views Last 7d,Price Comparisons Last 7d,Days Since Last Visit,Expected Margin,Completed Purchase
0,L149579,2026-04-24 10:21:00,thirdparty,SEO,mobile,direct,Tehran,Pasargad,cash,82,-3,7070000.0,5.3,0,1,0,3,2,0,23.5,205000.0,0
1,L107590,2026-07-07 12:07:00,thirdparty,Paid,mobile,direct,Mashhad,Asia,installment,72,18,4770000.0,7.6,0,1,0,4,5,1,2.4,124000.0,0
2,L121258,2026-05-17 15:09:00,thirdparty,SEO,mobile,other_partner,Tehran,Pasargad,cash,210,9,7570000.0,11.7,1,1,0,2,2,4,7.2,219500.0,0


## 1. Dataset structure

Each row is one lead. The columns fall into three semantic groups — and this grouping matters for the availability and leakage discussion:

| Group | Columns | Available at |
|---|---|---|
| **Lead identity / context** | `Product Type`, `Channel`, `Device`, `Partner`, `City`, `Insurance Company`, `Payment Type` | lead creation |
| **User behaviour in the funnel** | `Sessions Last 7d`, `Offer Views Last 7d`, `Price Comparisons Last 7d`, `Visited Offer Page`, `Days Since Last Visit`, `Minutes Since Abandonment`, `Incoming Call Last 24h`, `Has Previous Purchase` | scoring time (some are 7-day rolling windows) |
| **Offer / deal economics** | `Price`, `Discount Percent`, `Expected Margin`, `Days To Policy Expiry` | offer creation |

`Completed Purchase` is the target and `Expected Margin` is the business weight.

In [2]:
dd = pd.read_csv(DATA / "data_dictionary.csv", encoding="utf-8-sig")
info = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "n_unique": raw.nunique(),
    "n_missing": raw.isna().sum(),
    "pct_missing": (raw.isna().mean() * 100).round(2),
}).reset_index(names="column_name").merge(dd, on="column_name", how="left")
info

,column_name,dtype,n_unique,n_missing,pct_missing,description
0,Lead ID,str,50000,0,0.00,شناسه Lead
1,Created At,datetime64[us],44789,0,0.00,زمان ایجاد Lead
2,Product Type,str,2,0,0.00,نوع محصولی که کاربر دنبالشه
3,Channel,str,4,0,0.00,کانالی که Lead ازش وارد شده
4,Device,str,2,0,0.00,Device مورد استفاده کاربر
5,Partner,str,4,0,0.00,Partner یا منبع مرتبط با Lead
6,City,str,10,936,1.87,شهر کاربر
7,Insurance Company,str,7,0,0.00,شرکت بیمه انتخاب‌شده یا موردنظر کاربر
8,Payment Type,str,3,0,0.00,روش پرداخت انتخاب‌شده
9,Minutes Since Abandonment,int64,360,0,0.00,چند دقیقه از آخرین رها کردن فرایند خرید گذشته


## 2. Target: class imbalance

Overall conversion is ~9.2%. Not extreme imbalance, but enough that **accuracy is meaningless** and a default 0.5 threshold makes no sense.

In [3]:
n = len(raw); pos = int(raw[TARGET].sum())
print(f"rows={n:,}  positives={pos:,}  base rate={pos/n:.4%}  imbalance ratio≈1:{(n-pos)/pos:.1f}")

fig, ax = plt.subplots(figsize=(4.5, 3.2))
raw[TARGET].value_counts().sort_index().plot.bar(ax=ax, color=["#8892a4", "#2f6fb2"])
ax.set_xticklabels(["Not purchased (0)", "Purchased (1)"], rotation=0)
ax.set_ylabel("leads"); ax.set_title(f"Target balance — base rate {pos/n:.1%}")
save(fig, "01_target_balance");

rows=50,180  positives=4,605  base rate=9.1770%  imbalance ratio≈1:9.9


## 3. Data quality

Four real issues showed up. Each one needs an explicit decision.

### 3.1 Missing values

Three columns have missing values, all under 2%:

In [4]:
miss = raw.isna().sum(); miss = miss[miss > 0]
print(pd.DataFrame({"n": miss, "pct": (miss / n * 100).round(2)}))
print("\nPrice and Discount missing together:", int((raw["Price"].isna() & raw["Discount Percent"].isna()).sum()))

                    n   pct
City              936  1.87
Price             930  1.85
Discount Percent  542  1.08

Price and Discount missing together: 13


**Is it MCAR?** For `City` and `Discount Percent`, roughly yes. For `Price`, no: the missing rate in the `Referral` channel is ~2.4x the rest — meaning the missingness itself carries signal (offers are probably not always generated for that channel).

In [5]:
rows = []
for col in ["City", "Price", "Discount Percent"]:
    by_ch = raw.groupby("Channel")[col].apply(lambda s: s.isna().mean())
    rows.append(pd.Series(by_ch, name=col))
mm = pd.concat(rows, axis=1).mul(100).round(2)
print("missing % by Channel:\n", mm)

for col in ["City", "Price", "Discount Percent"]:
    m = raw[col].isna()
    print(f"{col:18s} conv when missing={raw.loc[m, TARGET].mean():.3%}  when present={raw.loc[~m, TARGET].mean():.3%}")

missing % by Channel:
           City  Price  Discount Percent
Channel                                
CRM       1.75   1.72              1.14
Paid      1.87   1.46              1.10
Referral  1.85   3.51              1.20
SEO       1.90   1.47              0.99
City               conv when missing=10.363%  when present=9.154%
Price              conv when missing=9.570%  when present=9.170%
Discount Percent   conv when missing=8.487%  when present=9.184%


**Decision:** do not drop rows (wasteful, and such rows will arrive in production too). Instead:
- `City` → an explicit `"Unknown"` level (it has a slightly *higher* conversion rate, so it carries information).
- `Price` / `Discount Percent` → impute with the median **within `Product Type`** (the two products have completely separate price distributions), plus a binary `price_is_missing` flag. The model can decide whether to use the flag.

### 3.2 Duplicate records

No row is a full duplicate, but 180 `Lead ID`s appear twice — and within each pair **every column is identical except `Created At`**, which differs by a few minutes.

In [6]:
print("exact duplicate rows:", int(raw.duplicated().sum()))
print("duplicated Lead IDs   :", int(raw["Lead ID"].duplicated().sum()))

dups = raw[raw["Lead ID"].duplicated(keep=False)].sort_values(["Lead ID", "Created At"])
varying = dups.groupby("Lead ID")[[c for c in raw.columns if c != "Lead ID"]].nunique()
print("\ncolumns that vary inside duplicate pairs:",
      list(varying.columns[(varying > 1).any()]))
gap = dups.groupby("Lead ID")["Created At"].agg(lambda s: (s.max() - s.min()).total_seconds() / 60)
print(f"time gap within pairs (minutes): median={gap.median():.0f}  max={gap.max():.0f}")
dups.head(4)[["Lead ID", "Created At", "Channel", "Price", TARGET]]

exact duplicate rows: 0
duplicated Lead IDs   : 180

columns that vary inside duplicate pairs: ['Created At']
time gap within pairs (minutes): median=8  max=14


,Lead ID,Created At,Channel,Price,Completed Purchase
1144,L100058,2026-08-01 17:03:00,Paid,6760000.0,0
19840,L100058,2026-08-01 17:10:00,Paid,6760000.0,0
34482,L100078,2026-07-25 19:18:00,Paid,8110000.0,0
38511,L100078,2026-07-25 19:30:00,Paid,8110000.0,0


Interpretation: these are **re-submissions** (the user resubmitted the form, or ingestion wrote it twice), not two independent leads.

**Decision:** deduplicate on `Lead ID`, keeping the **latest** record (the freshest state). This is only 0.4% of the data, but if left alone the same lead can land in both train and validation → **leakage**.

### 3.3 `Days To Policy Expiry` is negative

~14% of rows are negative (down to -20). This is **not a data error** — it means the user's current policy has already expired. And this group has the *highest* conversion rate.

In [7]:
neg = raw["Days To Policy Expiry"] < 0
print(f"negative: {neg.sum():,} ({neg.mean():.1%})  conv={raw.loc[neg, TARGET].mean():.2%}"
      f"  vs non-negative conv={raw.loc[~neg, TARGET].mean():.2%}")

negative: 7,156 (14.3%)  conv=13.09%  vs non-negative conv=8.53%


**Decision:** do not clip, do not treat as outliers. Keep the sign as an explicit feature (`is_expired`) because it encodes urgency.

### 3.4 Logical inconsistency between `Visited Offer Page` and `Offer Views Last 7d`

These two should agree, but ~21% of rows contradict each other:

In [8]:
a = (raw["Offer Views Last 7d"] > 0) & (raw["Visited Offer Page"] == 0)
b = (raw["Offer Views Last 7d"] == 0) & (raw["Visited Offer Page"] == 1)
print(f"views>0 but Visited=0 : {a.sum():,} ({a.mean():.1%})")
print(f"views=0 but Visited=1 : {b.sum():,} ({b.mean():.1%})")
print(f"total inconsistent     : {(a|b).mean():.1%}")

views>0 but Visited=0 : 7,581 (15.1%)
views=0 but Visited=1 : 2,896 (5.8%)
total inconsistent     : 20.9%


Interpretation: **different time windows**, not a bug. `Visited Offer Page` refers to the current session/lead, whereas `Offer Views Last 7d` is a 7-day rolling window. So `views>0, Visited=0` means "has seen offers before but didn't reach one this time", and `views=0, Visited=1` means "a new user who just saw an offer".

**Decision:** don't "fix" either one. Build an interaction feature from their four-way combination, because the combination itself is meaningful.

## 4. Conversion patterns

From here on we work on the deduplicated frame.

In [9]:
df = raw.sort_values("Created At").drop_duplicates("Lead ID", keep="last").copy()
print(f"{len(raw):,} → {len(df):,} rows after dedup on Lead ID")

def rate_table(frame, col, bins=None, min_count=200):
    key = pd.qcut(frame[col], bins, duplicates="drop") if bins else frame[col].fillna("Unknown")
    g = frame.groupby(key, observed=True)[TARGET].agg(n="count", rate="mean")
    g = g[g["n"] >= min_count] if bins is None else g
    g["lift"] = g["rate"] / frame[TARGET].mean()
    return g.sort_values("rate", ascending=False)

50,180 → 50,000 rows after dedup on Lead ID


### 4.1 Strongest signals: behavioural intent

Three behavioural features have the highest discriminative power, and all three are **monotonic**:

In [10]:
for col in ["Visited Offer Page", "Has Previous Purchase", "Incoming Call Last 24h"]:
    g = df.groupby(col)[TARGET].agg(n="count", rate="mean")
    print(f"\n{col}\n{g.assign(rate=lambda x: (x['rate']*100).round(2))}")
    print(f"  → lift: {g['rate'].iloc[1] / g['rate'].iloc[0]:.2f}x")


Visited Offer Page
                        n   rate
Visited Offer Page              
0                   15475   3.32
1                   34525  11.79
  → lift: 3.55x

Has Previous Purchase
                           n   rate
Has Previous Purchase              
0                      34614   6.96
1                      15386  14.14
  → lift: 2.03x

Incoming Call Last 24h
                            n   rate
Incoming Call Last 24h              
0                       46033   8.74
1                        3967  14.14
  → lift: 1.62x


In [11]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, col in zip(axes, ["Offer Views Last 7d", "Sessions Last 7d", "Price Comparisons Last 7d"]):
    g = df.groupby(col)[TARGET].agg(n="count", rate="mean")
    g = g[g["n"] >= 100]
    ax.plot(g.index, g["rate"] * 100, "o-", color="#2f6fb2")
    ax.axhline(df[TARGET].mean() * 100, ls="--", c="#b23f3f", lw=1, label="base rate")
    ax.set_xlabel(col); ax.set_title(col, fontsize=10)
axes[0].set_ylabel("conversion %"); axes[0].legend(fontsize=8)
fig.suptitle("Engagement depth vs conversion", y=1.04)
save(fig, "02_engagement_vs_conversion");

**Pattern:** engagement depth raises conversion almost linearly. `Offer Views Last 7d` goes from 5.9% (zero views) to 16.1% (4+ views) — about **2.7x**. `Sessions Last 7d` goes from 6.8% to 13.6%.

**But `Price Comparisons Last 7d` is essentially flat** (9.1% → 9.6%, correlation ~0.002). On the surface it looks like a purchase-intent feature, but it carries virtually no information — a drop candidate.

### 4.2 Time urgency: the strongest non-behavioural signal

Two time axes, both monotonically decaying — and these are the two main operational takeaways for the sales team.

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, col, nb in zip(axes, ["Minutes Since Abandonment", "Days To Policy Expiry"], [8, 8]):
    g = rate_table(df, col, bins=nb).sort_index()
    ax.bar(range(len(g)), g["rate"] * 100, color="#2f6fb2")
    ax.set_xticks(range(len(g)))
    ax.set_xticklabels([str(i) for i in g.index], rotation=45, ha="right", fontsize=7)
    ax.axhline(df[TARGET].mean() * 100, ls="--", c="#b23f3f", lw=1)
    ax.set_title(col, fontsize=10)
axes[0].set_ylabel("conversion %")
fig.suptitle("Urgency: both axes decay monotonically", y=1.04)
save(fig, "03_urgency_decay");

print(rate_table(df, "Minutes Since Abandonment", bins=6).sort_index(), "\n")
print(rate_table(df, "Days To Policy Expiry", bins=6).sort_index())

                              n      rate      lift
Minutes Since Abandonment                          
(0.999, 40.0]              8570  0.115753  1.262572
(40.0, 61.0]               8344  0.109060  1.189577
(61.0, 83.0]               8114  0.093419  1.018966
(83.0, 110.0]              8332  0.090614  0.988378
(110.0, 151.0]             8335  0.080864  0.882023
(151.0, 360.0]             8305  0.059603  0.650116 

                          n      rate      lift
Days To Policy Expiry                          
(-20.001, 1.0]         9094  0.130856  1.427307
(1.0, 7.0]             7725  0.108479  1.183235
(7.0, 13.0]            9585  0.089306  0.974108
(13.0, 18.0]           7787  0.081161  0.885263
(18.0, 25.0]           8410  0.073365  0.800230
(25.0, 60.0]           7399  0.060954  0.664858


**Pattern (the most operationally important finding):**

1. **`Minutes Since Abandonment` — the fast-call window burns down.** Leads less than 40 minutes past abandonment convert at 11.7%; above 151 minutes it drops to 6.0%. A **~2x decay**. So score alone is not enough; the call queue must also account for recency, because a lead's value decays with time.

2. **`Days To Policy Expiry` — proximity to expiry drives purchase.** ≤1 day (including already-expired) converts at 13.1% versus 6.1% for 25+ days. A **~2.1x spread**.

These two are roughly independent, so their effects are additive.

### 4.3 Offer economics: price, discount, and one trap

Discount has a monotonic upward effect. Price has a downward effect — **but that effect is mostly spurious.**

In [13]:
print(rate_table(df, "Discount Percent", bins=6).sort_index(), "\n")
print(rate_table(df, "Price", bins=6).sort_index())

print("\n— price effect within each product —")
for prod, sub in df.groupby("Product Type"):
    g = rate_table(sub, "Price", bins=5).sort_index()
    print(f"\n{prod} (median price={sub['Price'].median():,.0f}):")
    print(g[["n", "rate"]].assign(rate=lambda x: (x["rate"] * 100).round(2)))

                     n      rate      lift
Discount Percent                          
(-0.001, 2.3]     8440  0.084479  0.921451
(2.3, 4.1]        8162  0.080127  0.873990
(4.1, 5.6]        8614  0.088577  0.966151
(5.6, 7.0]        7770  0.090219  0.984062
(7.0, 8.9]        8507  0.100623  1.097546
(8.9, 18.0]       7966  0.106829  1.165238 



                             n      rate      lift
Price                                             
(1469999.999, 6030000.0]  8216  0.101996  1.112523
(6030000.0, 7280000.0]    8164  0.090519  0.987340
(7280000.0, 8470000.0]    8161  0.101336  1.105319
(8470000.0, 10130000.0]   8215  0.101400  1.106020
(10130000.0, 18070000.0]  8149  0.085532  0.932940
(18070000.0, 40720000.0]  8167  0.068691  0.749248



— price effect within each product —

carbody (median price=19,560,000):
                              n  rate
Price                                
(2639999.999, 14790000.0]  2723  8.70
(14790000.0, 18090000.0]   2717  7.18
(18090000.0, 20940000.0]   2717  6.59
(20940000.0, 24150000.0]   2714  7.07
(24150000.0, 40720000.0]   2714  6.85

thirdparty (median price=7,490,000):
                             n   rate
Price                                
(1469999.999, 5840000.0]  7126  10.57
(5840000.0, 6990000.0]    7090   8.82
(6990000.0, 8010000.0]    7113   9.77
(8010000.0, 9220000.0]    7105  10.44
(9220000.0, 15630000.0]   7053   9.80


**The trap:** across the full dataset the most expensive decile converts at 6.9% and the cheapest at 10.2% — suggesting "higher price = lower conversion". But the two products have completely separate price distributions (`carbody` median ~19.6M versus `thirdparty` ~7.5M) and `carbody` itself converts lower (7.3% vs 9.9%). **Within each product, the price effect largely disappears.** So raw `Price` is to a large extent a proxy for `Product Type`.

**Decision:** instead of raw price, build `price_percentile_within_product` so that the real "expensive relative to peers" effect is separated from the product-type effect.

### 4.4 `Expected Margin` is a deterministic function of price — not an independent feature

Worth checking explicitly, because it both creates collinearity and, if used wrongly, behaves like leakage.

In [14]:
ratio = (df["Expected Margin"] / df["Price"])
print(ratio.groupby(df["Product Type"]).describe()[["count", "mean", "std", "min", "max"]])
print(f"\ncorr(Price, Expected Margin) = {df['Price'].corr(df['Expected Margin']):.4f}")

fig, ax = plt.subplots(figsize=(5.5, 3.8))
for prod, sub in df.groupby("Product Type"):
    s = sub.sample(min(4000, len(sub)), random_state=0)
    ax.scatter(s["Price"] / 1e6, s["Expected Margin"] / 1e6, s=3, alpha=.3, label=prod)
ax.set_xlabel("Price (M)"); ax.set_ylabel("Expected Margin (M)")
ax.set_title("Margin is a fixed ~2.7% / ~3.9% band of Price"); ax.legend()
save(fig, "04_margin_vs_price");

                count      mean       std       min       max
Product Type                                                 
carbody       13585.0  0.038694  0.001771  0.035985  0.042008
thirdparty    35487.0  0.026690  0.001770  0.023976  0.030000

corr(Price, Expected Margin) = 0.9888


**Finding:** `Expected Margin` = Price x a narrow, product-dependent coefficient (`thirdparty` ~2.4–3.0%, `carbody` ~3.6–4.2%). Correlation with price is ~0.99.

**Decision:** do **not** feed `Expected Margin` to the model as an input feature (redundant with price, and it creates collinearity). Instead give it its proper role: the **business weight in the prioritization layer** — final priority based on `P(purchase) x Expected Margin` (expected value) rather than raw probability. That is what the sales team should actually be calling on.

### 4.5 Context features: weak but real signal

In [15]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6.5))
cats = ["Channel", "Payment Type", "Insurance Company", "Product Type", "Device", "City"]
for ax, col in zip(axes.ravel(), cats):
    g = rate_table(df, col).sort_values("rate")
    ax.barh(range(len(g)), g["rate"] * 100, color="#2f6fb2")
    ax.set_yticks(range(len(g))); ax.set_yticklabels(g.index, fontsize=8)
    ax.axvline(df[TARGET].mean() * 100, ls="--", c="#b23f3f", lw=1)
    ax.set_title(col, fontsize=10); ax.set_xlabel("conversion %", fontsize=8)
fig.suptitle("Categorical features (dashed = base rate)", y=1.01)
save(fig, "05_categorical_rates");

for col in ["Channel", "Payment Type", "Insurance Company"]:
    print(f"\n{col}:\n", rate_table(df, col).assign(rate=lambda x: (x['rate']*100).round(2), lift=lambda x: x['lift'].round(2)))


Channel:
               n   rate  lift
Channel                     
CRM        6026  10.95  1.19
SEO       19123   9.88  1.08
Referral   8770   8.55  0.93
Paid      16081   7.98  0.87



Payment Type:
                   n   rate  lift
Payment Type                    
bnpl           7035  10.65  1.16
installment   13892   9.59  1.05
cash          29073   8.61  0.94

Insurance Company:
                        n   rate  lift
Insurance Company                    
Saman               8444  10.62  1.16
Asia                9444   9.52  1.04
Alborz              4425   9.36  1.02
Pasargad            4926   9.32  1.02
Parsian             6044   9.22  1.01
Dana                6532   8.56  0.93
Iran               10185   7.84  0.86


**Pattern:** the range here is narrow (~8% to ~11%, i.e. lift roughly 0.9–1.2), but the direction is sensible:

- **`Channel`:** `CRM` highest (10.9%) — reasonable, these are our own known users. `Paid` lowest (8.0%) — bought traffic is colder.
- **`Payment Type`:** `bnpl` (10.6%) > `installment` (9.6%) > `cash` (8.6%) — reducing payment friction works.
- **`Insurance Company`:** `Saman` 10.6% versus `Iran` 7.8%.
- **`City`:** almost no signal (8.1–10.6% on small samples). With 11 levels and no real effect, a candidate for dropping or merging.

Bottom line: these features are not worthless, but they will not be the model's main signal. The main signal is **behaviour + urgency**.

### 4.6 Features on one scale: what actually discriminates

In [16]:
num_cols = ["Offer Views Last 7d", "Sessions Last 7d", "Minutes Since Abandonment",
            "Days To Policy Expiry", "Discount Percent", "Price", "Days Since Last Visit",
            "Price Comparisons Last 7d", "Expected Margin"]
bin_cols = ["Visited Offer Page", "Has Previous Purchase", "Incoming Call Last 24h"]

rows = []
for col in num_cols:
    g = rate_table(df, col, bins=5).sort_index()
    rows.append((col, g["rate"].max() / g["rate"].min(), abs(df[col].corr(df[TARGET]))))
for col in bin_cols:
    g = df.groupby(col)[TARGET].mean()
    rows.append((col, g.iloc[1] / g.iloc[0], abs(df[col].corr(df[TARGET]))))

strength = (pd.DataFrame(rows, columns=["feature", "max_min_lift", "abs_corr"])
              .sort_values("max_min_lift", ascending=False).reset_index(drop=True))
print(strength.round(3))

fig, ax = plt.subplots(figsize=(7, 4.2))
s = strength.sort_values("max_min_lift")
ax.barh(s["feature"], s["max_min_lift"], color="#2f6fb2")
ax.axvline(1, ls="--", c="#b23f3f", lw=1)
ax.set_xlabel("conversion lift (best bin / worst bin)")
ax.set_title("Univariate discriminative power")
save(fig, "06_feature_strength");

                      feature  max_min_lift  abs_corr
0          Visited Offer Page         3.549     0.136
1         Offer Views Last 7d         2.464     0.121
2       Days To Policy Expiry         2.106     0.078
3       Has Previous Purchase         2.033     0.115
4            Sessions Last 7d         2.000     0.080
5   Minutes Since Abandonment         1.910     0.066
6      Incoming Call Last 24h         1.618     0.051
7                       Price         1.517     0.039
8             Expected Margin         1.487     0.041
9            Discount Percent         1.304     0.032
10      Days Since Last Visit         1.140     0.014
11  Price Comparisons Last 7d         1.049     0.002


**Signal ranking:** `Visited Offer Page` (3.5x) → `Offer Views Last 7d` (2.7x) → `Days To Policy Expiry` (2.1x) → `Minutes Since Abandonment` (2.0x) → `Has Previous Purchase` (2.0x) → `Sessions Last 7d` (2.0x) → `Incoming Call Last 24h` (1.6x) → everything else ≈ 1.3x or below.

No feature has an implausibly large lift → **there is no obvious leakage in this data.** That is a good sign: the target is genuinely hard and the model has to aggregate weak signals. Expecting AUC around 0.68–0.72 is reasonable, not 0.95.

### 4.7 Signals are additive — and that is the whole business case

In [17]:
hot = ((df["Visited Offer Page"] == 1) & (df["Has Previous Purchase"] == 1)
       & (df["Days To Policy Expiry"] <= 7) & (df["Minutes Since Abandonment"] <= 60))
cold = ((df["Visited Offer Page"] == 0) & (df["Has Previous Purchase"] == 0)
        & (df["Days To Policy Expiry"] > 21))

for name, mask in [("HOT (engaged + returning + urgent + fresh)", hot),
                   ("COLD (no offer view + new + far expiry)", cold)]:
    print(f"{name}\n  n={mask.sum():,} ({mask.mean():.1%} of leads)  conv={df.loc[mask, TARGET].mean():.2%}"
          f"  lift={df.loc[mask, TARGET].mean()/df[TARGET].mean():.2f}x\n")

fig, ax = plt.subplots(figsize=(6, 3.2))
seg = pd.Series({"COLD": df.loc[cold, TARGET].mean(), "all leads": df[TARGET].mean(),
                 "HOT": df.loc[hot, TARGET].mean()}) * 100
seg.plot.barh(ax=ax, color=["#8892a4", "#c0c6d0", "#2f6fb2"])
ax.set_xlabel("conversion %"); ax.set_title("Simple rule-based segments already separate 5x")
save(fig, "07_segments");

HOT (engaged + returning + urgent + fresh)
  n=1,328 (2.7% of leads)  conv=25.38%  lift=2.77x

COLD (no offer view + new + far expiry)
  n=2,914 (5.8% of leads)  conv=1.92%  lift=0.21x



**Pattern:** a simple hand-written rule with no model at all isolates a segment worth ~2.7% of lead volume that converts at **25.3%** (2.8x the baseline), against a cold segment at ~5%.

That ~5x spread between the two extremes shows the **problem is modelable** and that the signals are reasonably independent and additive. It also gives us a **necessary baseline**: the ML model has to beat this hand-written rule by a meaningful margin, otherwise it isn't worth maintaining.

## 5. Temporal structure: the most important finding for validation

The data spans 5 months (1 April – 28 August 2026). Hour-of-day and day-of-week show **no** pattern at all (8.0–10.7%, pure noise). But the monthly trend has a real drift.

In [18]:
print("range:", df["Created At"].min(), "→", df["Created At"].max())
monthly = df.groupby(df["Created At"].dt.to_period("M"))[TARGET].agg(n="count", rate="mean")
print("\nmonthly:\n", monthly.assign(rate=lambda x: (x["rate"]*100).round(2)))

hourly = df.groupby(df["Created At"].dt.hour)[TARGET].mean()
dow = df.groupby(df["Created At"].dt.dayofweek)[TARGET].mean()
print(f"\nhour-of-day spread : {hourly.min():.2%} – {hourly.max():.2%}  (noise)")
print(f"day-of-week spread : {dow.min():.2%} – {dow.max():.2%}  (noise)")

weekly = df.groupby(df["Created At"].dt.to_period("W"))[TARGET].agg(n="count", rate="mean")
weekly = weekly[weekly["n"] > 500]
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(range(len(weekly)), weekly["rate"] * 100, "o-", color="#2f6fb2")
ax.axhline(df[TARGET].mean() * 100, ls="--", c="#b23f3f", lw=1, label="overall base rate")
ax.set_xticks(range(0, len(weekly), 2))
ax.set_xticklabels([str(weekly.index[i]).split("/")[0] for i in range(0, len(weekly), 2)],
                   rotation=45, ha="right", fontsize=7)
ax.set_ylabel("conversion %"); ax.set_title("Weekly conversion — sustained decline from August")
ax.legend(fontsize=8)
save(fig, "08_temporal_drift");

range: 2026-04-01 00:19:00 → 2026-08-28 23:58:00

monthly:
                 n  rate
Created At             
2026-04     10081  9.76
2026-05     10245  9.72
2026-06      9840  9.49
2026-07     10454  9.36
2026-08      9380  7.38

hour-of-day spread : 8.07% – 10.67%  (noise)
day-of-week spread : 8.28% – 10.18%  (noise)


**Critical finding:** conversion is stable from April through July (~9.3–9.8%), then drops to **7.4%** in August — a ~24% relative decline that persists across several consecutive weeks, so it is not noise.

**Two direct consequences:**

1. **The split must be temporal, not random.** With a random `train_test_split`, the model looks from the future into the past and the evaluation comes out optimistic. Split: train on April–July, validate/test on August — exactly how it works in production (train on the past, predict the future).

2. **The model's output probabilities will not stay calibrated.** Trained on a 9.5% period and scoring a 7.4% period, it will systematically over-predict. For *ranking* (our main use case) this matters less, but it breaks any absolute threshold or expected-value calculation → so monitoring on base rate plus periodic recalibration is required.

## 6. Summary: from finding to decision

### Data quality issues

| Issue | Scale | Decision |
|---|---|---|
| Missing `City` / `Price` / `Discount` | <2% | Impute (median within `Product Type`) + missing flag; `City` → `"Unknown"` level |
| `Price` missingness depends on `Channel` | 3.5% in Referral vs ~1.5% | Keep the missing flag — it is itself signal |
| Duplicate `Lead ID` | 180 pairs | Dedup keeping the latest record — prevents train/test leakage |
| Negative `Days To Policy Expiry` | 14% | Valid (expired policy). No clipping; add an `is_expired` feature |
| Offer-page inconsistency | 21% | Different time windows, not a bug. Build an interaction feature |

### Feature decisions

| Decision | Reason |
|---|---|
| Drop `Expected Margin` from model inputs | Deterministic function of price (corr 0.99); use it as the business weight in the prioritization layer instead |
| Drop or deprioritize `Price Comparisons Last 7d` | Lift ≈ 1.0, corr ≈ 0.002 — uninformative |
| `Price` → percentile within `Product Type` | The raw price effect is a proxy for product type |
| Deprioritize `City` | 11 levels, no real signal |
| Build urgency features (`is_expired`, recency buckets) | The strongest non-behavioural signals |
| Drop hour-of-day / day-of-week features | Pure noise |

### Validation and modeling decisions

- **Temporal split** (April–July train / August test) — mandatory given the August drift.
- **Metrics:** PR-AUC and **Precision@k / Lift@k** as the primary metrics (k = daily telesales capacity), because this is ranking under a capacity constraint. Accuracy is meaningless. ROC-AUC only as a secondary metric.
- **Class imbalance (1:9.9):** no heavy resampling. Tuning `scale_pos_weight` and choosing the threshold from capacity rather than 0.5 is sufficient.
- **Required baseline:** the hand-written rule from section 4.7 (2.8x lift). The model must beat it meaningfully.
- **Final priority is not raw probability.** Given that margin differs ~1.5x across products and lead value burns down with time, the ranking should be based on **`P(purchase) x Expected Margin`**, adjusted for recency.
- **Monitoring:** track the conversion base rate and the score distribution; the August drift shows this data is not stationary and periodic recalibration is needed.